# 01 · Data loading and label direction

**Goal:** produce one clean, trustworthy dataset that every later notebook uses.

All logic lives in `src/phishguard/data.py` (with tests in `tests/test_data.py`). This notebook shows the evidence behind each decision; the decisions themselves are logged in `docs/DECISIONS.md`.

Later notebooks only need:
```python
from phishguard.data import load_clean, TARGET
df = load_clean()
```

In [1]:
import pandas as pd
from phishguard import data

## 1. Load the raw file and check it is complete

PhiUSIIL is published with **235,795 rows × 55 columns**. Our first cleaning notebook only saw 196,403 rows because the Colab upload was cut off, and nothing complained. `load_raw()` now checks the shape and raises an error instead of continuing silently.

In [2]:
raw = data.load_raw()
raw.shape

(235795, 55)

## 2. Label direction

The dataset uses `label = 1` for **legitimate** and `label = 0` for **phishing**.

scikit-learn computes precision, recall and F1 for the positive class (`1`). We want those numbers to describe how well we catch **phishing**, so phishing must be `1`. `add_target()` creates `is_phishing` and drops `label`, so nobody can accidentally use the old encoding.

In [3]:
labelled = data.add_target(raw)
pd.crosstab(raw["label"], labelled[data.TARGET], rownames=["label (original)"], colnames=[data.TARGET])

is_phishing,0,1
label (original),,
0,0,100945
1,134850,0


## 3. What needs cleaning? Look before deciding

### 3a. Trailing slashes leak the label

In [4]:
ends_with_slash = labelled["URL"].str.endswith("/")
ends_with_slash.groupby(labelled[data.TARGET]).mean().rename("share of URLs ending in '/'")

is_phishing
0    0.000000
1    0.364684
Name: share of URLs ending in '/', dtype: float64

About 36% of phishing URLs end in `/` and **no** legitimate URL does. This is a side effect of how the two classes were collected, not something about phishing itself. A model would learn "ends in / → phishing", score well on the test set, and fail on real URLs. We remove the trailing slash.

### 3b. Duplicates and conflicting labels

In [5]:
normalised = labelled["URL"].str.strip().str.rstrip("/")
print("exact duplicate URLs:          ", labelled["URL"].duplicated().sum())
print("duplicates after removing '/': ", normalised.duplicated().sum())

labels_per_url = labelled.groupby(normalised)[data.TARGET].nunique()
conflicts = labels_per_url[labels_per_url > 1].index
labelled.loc[normalised.isin(conflicts), ["URL", data.TARGET]]

exact duplicate URLs:           425
duplicates after removing '/':  1590


,URL,is_phishing
36057,https://www.education.gouv.fr/,1
84900,https://www.education.gouv.fr,0


- Duplicate URLs must go. Otherwise the same URL can end up in both train and test, and test scores look better than they are.
- One government site is labelled both ways. That's label noise. We can't be sure which label is right, so we drop every copy.

## 4. Apply the cleaning

In [6]:
clean, report = data.clean(labelled)
report

{'rows_in': 235795,
 'conflicting_label_rows_dropped': 2,
 'duplicate_rows_dropped': 1589,
 'rows_out': 234204}

In [7]:
counts = clean[data.TARGET].value_counts().rename({0: "legitimate", 1: "phishing"})
pd.DataFrame({"count": counts, "percent": (counts / counts.sum() * 100).round(2)})

,count,percent
is_phishing,,
legitimate,134849,57.58
phishing,99355,42.42


The classes are mildly imbalanced (about 58/42). That doesn't need resampling, but we should report F1 and ROC-AUC alongside accuracy and use **stratified** splits.

## 5. Other shortcuts we noticed (for the feature-selection milestone)

We didn't fix these here, because what to do about them is a modelling decision for the team:

In [8]:
clean.groupby(data.TARGET)[["IsHTTPS", "URLSimilarityIndex"]].agg(lambda s: (s == s.max()).mean()).rename(
    columns={"IsHTTPS": "share with HTTPS", "URLSimilarityIndex": "share with similarity = 100"})

,share with HTTPS,share with similarity = 100
is_phishing,,
0,1.00000,1.000000
1,0.48842,0.007881


- **Every** legitimate URL uses HTTPS and has `URLSimilarityIndex = 100`. Either column alone nearly separates the classes.
- Several columns describe the downloaded web page (`LineOfCode`, `NoOfImage`, …), not the URL.

Next milestone: decide which columns a URL-only model may use, and split train/test by domain.